In [1]:
import os
import glob
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# --- CONFIGURATION ---
DATASET_PATH = r"../data/raw/Audio/Baby Cry Dataset/"
OUTPUT_PATH = r"../data/processed/"
output_x = os.path.join(OUTPUT_PATH, "X_images.npy")
output_y = os.path.join(OUTPUT_PATH, "y_labels.npy")

# Shape Settings (Must be exact!)
SAMPLE_RATE = 22050
DURATION = 5.0
SAMPLES = int(SAMPLE_RATE * DURATION) # 110,250 samples
N_MELS = 128
TIME_STEPS = 216  # Fixed width for 5s audio with hop_length=512

# Labels (3 Classes)
LABEL_MAP = {
    'hungry': 0,
    'belly pain': 1, 'cold_hot': 1, 'discomfort': 1, # Pain Group
    'tired': 2, 'burping': 2, 'lonely': 2, 'scared': 2 # Normal Group
}

def audio_to_image(file_path):
    try:
        # 1. Load Audio (Force 5s)
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
        
        # 2. Pad or Truncate to ensure exact length
        if len(y) < SAMPLES:
            y = np.pad(y, (0, SAMPLES - len(y))) # Pad with zeros
        else:
            y = y[:SAMPLES] # Cut extra
            
        # 3. Create Mel Spectrogram
        # This creates the "Image" (Height=128, Width=216)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
        mel_db = librosa.power_to_db(mel, ref=np.max) # Convert to Log Scale (Colors)
        
        # 4. Fix Shape (Ensure width is exactly TIME_STEPS)
        if mel_db.shape[1] < TIME_STEPS:
            mel_db = np.pad(mel_db, ((0, 0), (0, TIME_STEPS - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :TIME_STEPS]
            
        # 5. Add "Color Channel" (Required for CNN)
        # Shape becomes (128, 216, 1) -> Like a Grayscale Image
        return mel_db[..., np.newaxis] 
        
    except Exception as e:
        print(f"Error: {e}")
        return None

# --- MAIN LOOP ---
X_data = []
y_data = []

print("📸 Starting Visual Data Processing...")

for folder_name, label in LABEL_MAP.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)
    files = glob.glob(os.path.join(folder_path, "*.wav"))
    
    print(f"   Processing {folder_name} ({len(files)} files)...")
    
    for file in files:
        img = audio_to_image(file)
        if img is not None:
            # ORIGINAL
            X_data.append(img)
            y_data.append(label)
            
            # AUGMENTATION (Simple copy for Normal/Hunger to balance)
            # You can add more advanced augmentation later (Time Masking)
            if label in [0, 2]: # Balance Hunger & Normal
                X_data.append(img) # Simple 2x duplication
                y_data.append(label)

X = np.array(X_data)
y = np.array(y_data)

# Normalize Data (0 to 1) - Critical for Neural Networks
X = (X - X.min()) / (X.max() - X.min())

# One-Hot Encode Labels (0 -> [1,0,0])
y = to_categorical(y, num_classes=3)

print(f"\n✅ Done! Saving Data...")
print(f"Images Shape: {X.shape}") # Should be (N, 128, 216, 1)
print(f"Labels Shape: {y.shape}")

np.save(output_x, X)
np.save(output_y, y)
print("💾 Data Saved to .npy files!")

f:\Research\Project\Final\infant-growth-monitoring-system\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


📸 Starting Visual Data Processing...
   Processing hungry (382 files)...
   Processing belly pain (126 files)...
   Processing cold_hot (107 files)...
   Processing discomfort (138 files)...
   Processing tired (136 files)...
   Processing burping (118 files)...
   Processing lonely (11 files)...
   Processing scared (20 files)...

✅ Done! Saving Data...
Images Shape: (1705, 128, 216, 1)
Labels Shape: (1705, 3)
💾 Data Saved to .npy files!
